In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [18]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [19]:
X_train = pd.read_csv("../../data/processed/X_train.csv")
X_test = pd.read_csv("../../data/processed/X_test.csv")
y_train = pd.read_csv("../../data/processed/y_train.csv", header=None).squeeze()
y_test = pd.read_csv("../../data/processed/y_test.csv", header=None).squeeze()

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
top_5_cities = X_train["city"].value_counts().head(5).index.tolist()

encoder = OneHotEncoder(
    categories=[top_5_cities], handle_unknown="ignore", sparse_output=False
)

numerical_columns = X_train.drop(columns=["city"]).columns  
preprocessor = ColumnTransformer(
    transformers=[
        ("city", encoder, ["city"]),
        ("num", StandardScaler(), numerical_columns),
    ]
)

In [21]:
base_models = {
    "LinearRegression": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]),
     "Ridge": Pipeline([
        ("preprocessor", preprocessor),
        ("model", Ridge(alpha=1.0))
    ]),
     "Lasso": Pipeline([
        ("preprocessor", preprocessor),
        ("model", Lasso(alpha=1.0))
    ]),
     "RandomForestRegressor": Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(random_state=42))
    ]),
     "XGBRegressor": Pipeline([
        ("preprocessor", preprocessor),
        ("model", XGBRegressor(random_state=42))
    ])
}

In [22]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

results = []
for model_name, model in base_models.items():
     model.fit(X_train, y_train)
     y_pred_model = model.predict(X_test)
     mse = mean_squared_error(y_test, y_pred_model)
     mae = mean_absolute_error(y_test, y_pred_model)
     r2 = r2_score(y_test, y_pred_model)
     results.append({
          "Name" : model_name,
          "MSE" : mse,
          "MAE" : mae,
          "r2_score" : r2
          })

In [23]:
experiment_results = pd.DataFrame(results).round(2)
experiment_results

,Name,MSE,MAE,r2_score
0,LinearRegression,2.520356e+10,116377.83,0.64
1,Ridge,2.521272e+10,116405.42,0.64
2,Lasso,2.520372e+10,116378.43,0.64
3,RandomForestRegressor,2.622018e+10,116534.21,0.63
4,XGBRegressor,2.859463e+10,120901.94,0.59


In [24]:
import os 
os.makedirs('../../reports', exist_ok=True)

# Save Experiment results
experiment_results.to_csv('../../reports/01_experiment_results.csv', index=False)
print("Results saved to reports/01_experiment_results.csv")

Results saved to reports/01_experiment_results.csv
